In [ ]:
# Run this cell first — installs required packages
!pip install gstools tifffile -q

In [ ]:
# Mount Google Drive — your files will be saved here so nothing is lost between sessions
from google.colab import drive
drive.mount('/content/drive')

# All outputs saved here — create it if it doesn't exist
from pathlib import Path
DRIVE_DIR  = Path('/content/drive/MyDrive/pgsca_cache')
DRIVE_DIR.mkdir(exist_ok=True)
PARAMS_PATH = DRIVE_DIR / 'avg_params.json'
TABLE_PATH  = DRIVE_DIR / 'kmap_table.npy'
print(f'Cache directory: {DRIVE_DIR}')

In [ ]:
# Upload your files using the panel below.
# You need to upload:
#   1. ThreePhase.tif          (your data file)
#   2. pgsca/hybrid_tools.py
#   3. pgsca/karnaugh_tools.py
#   4. pgsca/pgs_tools.py
#   5. pgsca/__init__.py
from google.colab import files
import os

os.makedirs('/content/pgsca', exist_ok=True)
print('Upload ThreePhase.tif and all four pgsca/*.py files when the dialog opens.')
uploaded = files.upload()

# Move pgsca files into the right place
import shutil
for fname in uploaded:
    if fname.endswith('.py'):
        shutil.move(fname, f'/content/pgsca/{fname}')
    elif fname.endswith('.tif'):
        shutil.move(fname, f'/content/{fname}')

DATA_PATH = Path('/content/ThreePhase.tif')
print(f'\nData file exists: {DATA_PATH.exists()}')
print(f'pgsca files: {list(Path("/content/pgsca").glob("*.py"))}')

In [ ]:
import sys, time, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import tifffile

sys.path.insert(0, '/content')

from pgsca.hybrid_tools import recover_plurigaussian_multi, make_pgs_seed
from pgsca.karnaugh_tools import build_table, sequential_simulate

lut = np.zeros(256, int)
lut[0], lut[128], lut[255] = 0, 1, 2
vol = lut[tifffile.imread(DATA_PATH)]
print(f'Volume loaded: {vol.shape}   unique phases: {np.unique(vol)}')

In [ ]:
# Reduced slice counts — fast enough to run in Colab in ~30-40 mins
# Increase step size (e.g. range(0,256,3)) for even faster but lower quality
RECOVERY_INDICES     = list(range(0, 256, 5))    # ~51 slices for PGS averaging
TRAIN_INDICES        = list(range(2, 256, 5))    # ~51 slices for K-map
COMPARE_REAL_INDICES = [10, 50, 90, 130, 170, 210]
TIME_BUDGET      = None   # no limit — cache means we only do this once
TARGET_IDX       = 128
CA_PASSES        = 20
RNG_SEED         = 42
CMAP             = 'copper'
PHASES           = ['Phase 0', 'Phase 1', 'Phase 2']

def _legend(ax, loc='lower right'):
    cm = plt.colormaps[CMAP]
    ax.legend(handles=[Patch(color=cm(k/2), label=PHASES[k]) for k in range(3)],
              loc=loc, fontsize=9)

print(f'Recovery slices : {len(RECOVERY_INDICES)}')
print(f'Training slices : {len(TRAIN_INDICES)}')

In [ ]:
if PARAMS_PATH.exists():
    with open(PARAMS_PATH) as f:
        avg_params = json.load(f)
    avg_params['params_1'] = tuple(avg_params['params_1'])
    avg_params['params_2'] = tuple(avg_params['params_2'])
    n = avg_params['n_slices_used']
    print(f'Loaded cached PGS parameters ({n} slices) from Drive')
else:
    print(f'Recovering PGS parameters from {len(RECOVERY_INDICES)} slices ...\n')
    pgs_slices = [vol[i] for i in RECOVERY_INDICES]
    t0 = time.perf_counter()
    avg_params = recover_plurigaussian_multi(pgs_slices, time_budget=TIME_BUDGET)
    elapsed = time.perf_counter() - t0
    n = avg_params['n_slices_used']
    print(f'Done — {n} slice(s) processed in {elapsed:.1f}s')

    save_data = dict(avg_params)
    save_data['params_1'] = list(avg_params['params_1'])
    save_data['params_2'] = list(avg_params['params_2'])
    with open(PARAMS_PATH, 'w') as f:
        json.dump(save_data, f, indent=2)
    print(f'Saved to Google Drive')

In [ ]:
a1, Lmaj1, Lmin1 = avg_params['params_1']
a2, Lmaj2, Lmin2 = avg_params['params_2']

print('=' * 52)
print(f'  AVERAGED PGS PARAMETERS  ({n} slices)')
print('=' * 52)
print(f'  Proportions  : {np.round(avg_params["proportions"], 4)}')
print(f'  cut_1        : {avg_params["cut_1"]:.4f}')
print(f'  cut_2        : {avg_params["cut_2"]:.4f}')
print(f'\n  Field 1:')
print(f'    alpha  = {np.degrees(a1):.1f}°')
print(f'    L_maj  = {Lmaj1:.2f}')
print(f'    L_min  = {Lmin1:.2f}')
print(f'\n  Field 2:')
print(f'    alpha  = {np.degrees(a2):.1f}°')
print(f'    L_maj  = {Lmaj2:.2f}')
print(f'    L_min  = {Lmin2:.2f}')
print('=' * 52)

In [ ]:
if TABLE_PATH.exists():
    table = np.load(TABLE_PATH)
    print(f'Loaded cached K-map table from Drive  shape={table.shape}')
else:
    print(f'Building Moore K-map from {len(TRAIN_INDICES)} slices ...')
    train_slices = [vol[i] for i in TRAIN_INDICES]
    t0 = time.perf_counter()
    table = build_table([(s, s) for s in train_slices], neighbourhood='moore')
    print(f'Done in {time.perf_counter() - t0:.1f}s')
    np.save(TABLE_PATH, table)
    print(f'Saved to Google Drive')

n_seen = (table.sum(axis=1) > 0).sum()
print(f'K-map ready — {n_seen} / {table.shape[0]} neighbourhood patterns observed')

In [ ]:
print('Generating PGS seed ...')
seed = make_pgs_seed(
    shape=(256, 256),
    params_1=avg_params['params_1'],
    params_2=avg_params['params_2'],
    proportions=avg_params['proportions'],
    seed_1=7, seed_2=13,
)

target_slice = vol[TARGET_IDX]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(target_slice, cmap=CMAP, vmin=0, vmax=2)
axes[0].set_title(f'Real slice (index {TARGET_IDX})', fontsize=13)
axes[0].axis('off')
_legend(axes[0])
axes[1].imshow(seed, cmap=CMAP, vmin=0, vmax=2)
axes[1].set_title('PGS seed (averaged parameters)', fontsize=13)
axes[1].axis('off')
_legend(axes[1])
plt.suptitle('PGS seed vs. real slice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print(f'Running CA (up to {CA_PASSES} passes) ...')
t0 = time.perf_counter()
final_grid, history = sequential_simulate(
    table=table,
    shape=(256, 256),
    proportions=avg_params['proportions'],
    n_passes=CA_PASSES,
    improvement_tol=0.0005,
    rng=np.random.default_rng(RNG_SEED),
    initial_grid=seed,
)
print(f'CA finished in {time.perf_counter() - t0:.1f}s  ({len(history)} pass(es) run)')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(history)+1), history, marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('Pass'); ax.set_ylabel('Fraction changed (δ)')
ax.set_title('CA convergence'); ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, title in zip(axes,
    [target_slice, seed, final_grid],
    [f'Real slice (index {TARGET_IDX})', 'PGS seed (macro only)', 'CA-refined synthetic']):
    ax.imshow(img, cmap=CMAP, vmin=0, vmax=2)
    ax.set_title(title, fontsize=13)
    ax.axis('off')
cm = plt.colormaps[CMAP]
fig.legend(handles=[Patch(color=cm(k/2), label=PHASES[k]) for k in range(3)],
           loc='lower center', ncol=3, fontsize=11, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Real  →  PGS seed  →  CA-refined synthetic', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

In [ ]:
real_images = [vol[i] for i in COMPARE_REAL_INDICES]
all_images  = real_images + [final_grid]
synth_src   = len(real_images)

rng_disp    = np.random.default_rng(0)
order       = rng_disp.permutation(len(all_images))
synth_panel = int(np.where(order == synth_src)[0][0])
labels      = [chr(65 + i) for i in range(len(all_images))]

ncols = 4
nrows = int(np.ceil(len(all_images) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*5))
axes = axes.ravel()
for plot_pos, src_idx in enumerate(order):
    axes[plot_pos].imshow(all_images[src_idx], cmap=CMAP, vmin=0, vmax=2)
    axes[plot_pos].set_title(labels[plot_pos], fontsize=15, fontweight='bold')
    axes[plot_pos].axis('off')
for ax in axes[len(all_images):]:
    ax.set_visible(False)
cm = plt.colormaps[CMAP]
fig.legend(handles=[Patch(color=cm(k/2), label=PHASES[k]) for k in range(3)],
           loc='lower center', ncol=3, fontsize=12, bbox_to_anchor=(0.5, 0.01))
plt.suptitle('Can you spot the synthetic image?', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

print(f'\nAnswer: panel  {labels[synth_panel]}  is the synthetic image.')
print(f'Real slice indices shown: {COMPARE_REAL_INDICES}')